In [1]:
import tensorflow as tf
from tensorflow.keras.applications import *
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.datasets import cifar100
import numpy as np
from tensorflow.keras.applications.imagenet_utils import preprocess_input
import time

# Load CIFAR-100 dataset
(x_train, y_train), (x_test, y_test) = cifar100.load_data()

# Select 20 classes
selected_classes = list(range(20))
train_mask = np.isin(y_train, selected_classes).flatten()
test_mask = np.isin(y_test, selected_classes).flatten()

x_train_20 = x_train[train_mask]
y_train_20 = y_train[train_mask]
x_test_20 = x_test[test_mask]
y_test_20 = y_test[test_mask]

# Remap labels to 0-19
label_map = {cls: i for i, cls in enumerate(selected_classes)}
y_train_20 = np.array([label_map[y[0]] for y in y_train_20])
y_test_20 = np.array([label_map[y[0]] for y in y_test_20])

# One-hot encode labels
y_train_20_cat = tf.keras.utils.to_categorical(y_train_20, 20)
y_test_20_cat = tf.keras.utils.to_categorical(y_test_20, 20)

# Resize images to 224x224
x_train_20_resized = tf.image.resize(x_train_20, (224, 224)).numpy()
x_test_20_resized = tf.image.resize(x_test_20, (224, 224)).numpy()

# Preprocess images
x_train_20_resized = preprocess_input(x_train_20_resized)
x_test_20_resized = preprocess_input(x_test_20_resized)

# List of model constructors and names
model_constructors = [
    (Xception, "Xception"),
    (ResNet50, "ResNet50"),
    (MobileNet, "MobileNet"),
    (MobileNetV2, "MobileNetV2"),
    (DenseNet121, "DenseNet121"),
    (NASNetMobile, "NASNetMobile"),
    (EfficientNetB0, "EfficientNetB0"),
    (EfficientNetB1, "EfficientNetB1"),
    (EfficientNetB3, "EfficientNetB3"),
    (InceptionV3, "InceptionV3")
]

# Loop through each model
for model_fn, model_name in model_constructors:
    print(f"\nTraining {model_name}...")

    input_tensor = Input(shape=(224, 224, 3))
    base_model = model_fn(weights='imagenet', include_top=False, input_tensor=input_tensor)
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    predictions = Dense(20, activation='softmax')(x)
    model = Model(inputs=base_model.input, outputs=predictions)

    for layer in base_model.layers:
        layer.trainable = False

    model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])

    start_time = time.time()
    model.fit(x_train_20_resized, y_train_20_cat, epochs=1, batch_size=32, verbose=1)
    end_time = time.time()

    loss, accuracy = model.evaluate(x_test_20_resized, y_test_20_cat, verbose=0)
    print(f"{model_name} - Test accuracy: {accuracy:.4f}, Time taken: {end_time - start_time:.2f} seconds")


2025-10-11 00:41:57.793393: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760121717.944872   16113 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760121717.988489   16113 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-10-11 00:41:58.355738: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1760121726.229272   16113 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 72


Training Xception...


2025-10-11 00:42:29.631902: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 6021120000 exceeds 10% of free system memory.
2025-10-11 00:42:48.808912: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 6021120000 exceeds 10% of free system memory.
I0000 00:00:1760121777.640569   16156 service.cc:148] XLA service 0x7da2ac002ec0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1760121777.642607   16156 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce GTX 1070, Compute Capability 6.1
2025-10-11 00:42:57.873861: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1760121778.826831   16156 cuda_dnn.cc:529] Loaded cuDNN version 90501
2025-10-11 00:43:00.811833: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:306] Allocator (GPU_0_bfc) ran out of memor

  1/313 ━━━━━━━━━━━━━━━━━━━━ 1:03:41 12s/step - accuracy: 0.0625 - loss: 11.0826

I0000 00:00:1760121785.425379   16156 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


...

313/313 ━━━━━━━━━━━━━━━━━━━━ 45s 88ms/step - accuracy: 0.6333 - loss: 1.2710
ResNet50 - Test accuracy: 0.8520, Time taken: 126.08 seconds

Training MobileNet...


/tmp/ipykernel_16113/3702654391.py:60: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = model_fn(weights='imagenet', include_top=False, input_tensor=input_tensor)


313/313 ━━━━━━━━━━━━━━━━━━━━ 21s 39ms/step - accuracy: 0.2664 - loss: 2.4420
MobileNet - Test accuracy: 0.5080, Time taken: 88.60 seconds

Training MobileNetV2...


/tmp/ipykernel_16113/3702654391.py:60: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = model_fn(weights='imagenet', include_top=False, input_tensor=input_tensor)


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.2428 - loss: 2.5413
MobileNetV2 - Test accuracy: 0.4645, Time taken: 84.73 seconds

Training DenseNet121...
313/313 ━━━━━━━━━━━━━━━━━━━━ 68s 110ms/step - accuracy: 0.1415 - loss: 5.3959
DenseNet121 - Test accuracy: 0.3110, Time taken: 142.43 seconds

Training NASNetMobile...
313/313 ━━━━━━━━━━━━━━━━━━━━ 67s 99ms/step - accuracy: 0.1006 - loss: 4.1853
NASNetMobile - Test accuracy: 0.1800, Time taken: 127.38 seconds

Training EfficientNetB0...
313/313 ━━━━━━━━━━━━━━━━━━━━ 49s 76ms/step - accuracy: 0.3581 - loss: 2.1802
EfficientNetB0 - Test accuracy: 0.6250, Time taken: 126.98 seconds

Training EfficientNetB1...
313/313 ━━━━━━━━━━━━━━━━━━━━ 68s 108ms/step - accuracy: 0.1179 - loss: 6.2041
EfficientNetB1 - Test accuracy: 0.1955, Time taken: 143.91 seconds

Training EfficientNetB3...
313/313 ━━━━━━━━━━━━━━━━━━━━ 82s 136ms/step - accuracy: 0.2395 - loss: 2.7160
EfficientNetB3 - Test acc